**Volume 1 workbooks v0.1.0, published 2026-08-19.**


In [ ]:
import numpy as np
import pandas as pd
from IPython.display import display

try:
    np.set_printoptions(legacy='1.25')
except TypeError:
    pass
np.set_printoptions(suppress=True)
pd.set_option('display.float_format', '{:,.10g}'.format)

import pandas as pd

from ada_fsaudit_bridge import (
    att_sample,
    configure_environment,
    cvs_sample,
    load_dataset,
    lower_bound,
    mus_sample,
    set_notebook_context,
    upper_bound,
)
from scipy.stats import hypergeom

ADA_WORKSHOP_ID = "hypothesis-testing"
ADA_CHAPTER = "4"
ADA_NOTEBOOK_SOURCE = "notebooks/support/hypothesis-testing/support.Rmd"

def ada_set_context(exercise_ref: str) -> None:
    set_notebook_context(
        chapter=ADA_CHAPTER,
        exercise=exercise_ref,
        notebook=f'{ADA_WORKSHOP_ID}:{ADA_NOTEBOOK_SOURCE}',
    )

configure_environment()


# Hypothesis testing


## Exercise 4.1. Effect of the critical region on the sample size


This exercise examines how the critical region changes the amount of evidence needed for an attribute test. The practical task is to see how allowing more sample deviations changes the planned sample size before any fieldwork is performed.
The results on page 148 ($n = 45$ and $n = 102$) are obtained with the `FSaudit` package, first by creating an attribute object, filling it with the parameters of the test, and then applying the `size()` function.
The relevant parameters are the significance level `alpha`, the number of deviations in the population $M$ `popdev`, the population size $N$ `popn`, and the critical region `c`.


In [ ]:
ada_set_context("4.1")
mySample = att_sample(alpha=0.1, popdev=60, popn=1200, c=0).size()
mySample.n


With `c = 0`, the plan rejects only when no deviations are found. The sample size increases as we increase the critical region to `c = 2`, because the design must still control the risk of drawing a misleading conclusion while tolerating more observed deviations.


In [ ]:
ada_set_context("4.1")
mySample.size(c=2).n


This result is obtained using the default Hypergeometric distribution. To obtain the sample size with the binomial approximation, as in Table 4.3, we specify the distribution when setting up the attribute object. The binomial approximation ignores the finite population size $N$ and treats the draws as independent; compared with the hypergeometric distribution, this slightly increases the required sample size. The `tdr` argument is the tolerable deviation rate of the test: with $M = 60$ deviations in a population of $N = 1,200$ items, the tolerable deviation rate is $t_{dr} = 60 / 1,200 = 0.05$.


In [ ]:
ada_set_context("4.1")
mySample2 = att_sample(alpha=0.1, tdr=0.05, c=2, dist="binom").size()
mySample2.n


Compare the hypergeometric and binomial sample sizes. The learning point is that distribution choice is not just syntax: it affects the evidence required to support the planned test conclusion.


## Exercise 4.2. Significance levels


Here we read significance levels as evidence measures. The calculations ask how
probable it is to observe the sample result or a more favorable result if the
population contains the stated number of deviations.
On page 148 we calculated significance levels for the occurrence of finding one
or two errors. These probabilities are calculated in Python with the following
code:


In [ ]:
display(hypergeom.cdf(1, 1200, 60, 45))
display(hypergeom.cdf(2, 1200, 60, 102))


The first call gives the probability of finding one or fewer errors in a sample of 45; the second gives the probability of finding two or fewer errors in a sample of 102. After running the code, interpret the probabilities before looking up the formula details. Smaller values provide stronger evidence against the population deviation level used in the calculation.
Refer to Equation 1.1. Remember that Python's hypergeometric distribution
function uses the population size $N$, the number of deviations in the
population $M$, the sample size $n$, and the number of errors found $k$.


## Exercise 4.3. Type II error


On page 148 we also calculated the Type II error, for a scenario with no errors allowed in the sample and 24 errors in the population, or a population error rate of $24 / 1200 = 0.02$.


In [ ]:
hypergeom.sf(0, 1200, 24, 45)


Read this output as the risk that the sample does not support rejection even though the population contains the deviation rate specified in the scenario. This keeps the focus on what the test can miss, not only on what it can detect.


## Exercise 4.4. One-sided upper bounds


This exercise converts sample findings into one-sided upper bounds for the population deviation rate. The one-sided upper bounds $p_U$ in Table 4.1 are obtained as follows:


In [ ]:
display(upper_bound(k=0, popn=1200, n=102, alpha=0.10) / 1200)
display(upper_bound(k=3, popn=1200, n=102, alpha=0.10) / 1200)


Compare the two bounds. Finding more errors in the same sample increases the upper bound, which means more possible population deviation remains after the test.


## Exercise 4.5. Case: European innovation subsidies


This exercise applies the critical-region idea to the *Case: European innovation subsidies*. The sample size depends on whether the auditor requires a sample with no errors or allows a small number of errors before rejecting the null hypothesis.
When the null hypothesis $H_0 : M \geq 120,000$ is rejected when the sample yields no errors, the critical region is $\{k | k = 0\}$. We first create an `mus_obj` object, and load it with the parameters.


In [ ]:
ada_set_context("4.5")
subsidies = mus_sample(cl=0.95, popBv=12000000, pm=120000).size(ee=0)
subsidies.n


This result is exactly equal to that of the fixed-attribute sample. The comparison checks whether the monetary-unit planning problem behaves like the equivalent attribute test in this all-or-nothing error setting:


In [ ]:
ada_set_context("4.5")
myAttSample = att_sample(alpha=0.05, popn=12000000, popdev=120000).size(c=0)
myAttSample.n


To build a margin for one error, we may increase the critical region to $\{k | k \leq 1\}$, resulting in a sample size of $n =$ 473.


In [ ]:
ada_set_context("4.5")
myAttSample.size(c=1).n


The larger sample size is the cost of allowing one observed error while keeping the planned risk under control. In MUS, we increase the critical region by anticipating on the expected error (in monetary terms) in the population. Thus, in applications where selected items are either completely correct or completely incorrect, it is easier to calculate the minimum required sample size using the `att_obj` than using the `mus_obj`.


## Exercise 4.6. Case: Accounts receivable circularization


This exercise plans a monetary-unit sample for the accounts receivable case.
The practical task is to connect the population book value, performance
materiality, and expected error to the sample size required for Stringer
evaluation.
We start by setting up the MUS object `ar` (for Accounts Receivable) and first
verify that the number of sampling units `popn` and the total book value
`popBv` of the sampling frame match those in the population. Notice that these
statistics are calculated as soon as the object is loaded with the detail
amounts.


In [ ]:
ada_set_context("4.6")
accounts_receivable = load_dataset("accounts_receivable")
ar = mus_sample(
  bv=accounts_receivable["amount"],
  id=accounts_receivable["invoice"],
)
ar.popn, ar.popBv


Sample size calculation is invoked with the relevant values from the *Case: Accounts receivable circularization*.


In [ ]:
ada_set_context("4.6")
ar.size(cl=0.95, pm=450000, ee=100000, evalMeth="Stringer").n


Interpret the resulting sample size as the fieldwork effort implied by this MUS design. We then compare this with the sample size calculated using fixed-attribute sampling.


In [ ]:
ada_set_context("4.6")
ar2 = att_sample(alpha=0.05, popn=13500000, popdev=450000).size(c=1)
display(ar2.n)
display(ar2.size(c=2).n)


We can therefore infer that with an expected error of `ee = 100000`, we can tolerate between one and two 100% errors. The learning point is to translate a monetary planning assumption into an equivalent attribute-sampling tolerance, rather than treating the sample size as a standalone number.


## Exercise 4.7. Multiple hits with random selection


This exercise shows a practical sampling issue in MUS: large monetary items can be selected more than once when random selection is used. The goal is to recognize why high-value items need special attention in the sample output.
If we draw a sample of size $n = 1,100$ from the `accounts_receivable` population, the largest sampling units have an almost 100% inclusion probability and with a selection method such as `random` are likely to be hit more than once. The size of $n = 1,100$ is a deliberate demonstration choice for this exercise: with a population book value of 13,500,000 it implies a sampling interval of $13,500,000 / 1,100 \approx 12,273$, so book values approaching this interval are selected with near-certainty.
For example, the data file `accounts_receivable`, that comes with the `FSaudit` package, has the following six largest amounts:


In [ ]:
ada_set_context("4.7")
accounts_receivable.sort_values("amount", ascending=False).iloc[:6, :3]


We set up a new `mus_obj` and use a performance materiality of `pm = 36730` (36,730 euros). This is not the 450,000 performance materiality of the accounts receivable case in Exercise 4.6: the value was chosen by trial and error precisely so that the planned sample size equals the demonstration size of $n = 1,100$.


In [ ]:
ada_set_context("4.7")
multiple = mus_sample(
  bv=accounts_receivable["amount"],
  id=accounts_receivable["invoice"],
  pm=36730,
).size()
multiple.n


The sample size is intentionally large enough to make repeated selection of high-value invoices visible. We select the sample with the selection method `selMeth = "random"` and order it in decreasing order, displaying the six largest book values.


In [ ]:
ada_set_context("4.7")
multiple.select(selMeth="random", seed=1)
sample = multiple.sample
sample.sort_values("bv", ascending=False).head(6)


This example demonstrates that invoices 201719763 and 201710344 were selected twice. In practice, this is a cue to check how the method represents high-value items before interpreting the sample as a simple list of unique invoices.


## Exercise 4.8. Stringer bound


We continue the accounts receivable case by selecting the sample with the "randomized fixed" selection method. The goal is to create the sample that will later be evaluated after audit values are supplied.


In [ ]:
ada_set_context("4.8")
ar.select(selMeth="randomized.fixed", seed=345)
ar.sample.head(6)


Before we can evaluate the sample, we must first provide audit values. These should be provided in a list of the same order as the list of sample book values. We first copy the list of book values into a data frame, so correct items initially have audit value equal to book value (`av = ar$sample$bv`); only the items that are in error need to be overwritten with their audit values.


In [ ]:
ada_set_context("4.8")
myResults = ar.sample.loc[:, ["item", "bv"]].rename(columns={"bv": "av"}).copy()
myResults


Table 4.4 lists the three invoices (rows 16, 52, and 124 of the sample) that are assumed to be in error; their book and audit values are those of the three errors in that table.


In [ ]:
ada_set_context("4.8")
myResults.iloc[[15, 51, 123], :]


We update the audit values of the erroneous items. This step turns the selected sample into evaluated audit evidence.


In [ ]:
ada_set_context("4.8")
myResults.iloc[15, 1] = 4438.82
myResults.iloc[51, 1] = 0
myResults.iloc[123, 1] = 5531.38
myResults.iloc[[15, 51, 123], :]


The updated list of audit values is then submitted to the MUS object for evaluation.


In [ ]:
ada_set_context("4.8")
ar.evaluate(av=myResults["av"], evalMeth="stringer")


The results of the Stringer bound evaluation as presented in Table 4.6 and Table 4.7 are stored in the `Precision calculation` attribute. Its columns are, per error found: the tainting $t$, the point estimate $t X / n$, the precision gap widening $PGW$, and the tainting-weighted precision $t \times PGW$; the precision column sums to 354,168, the precision achieved of the Stringer bound evaluation.


In [ ]:
ada_set_context("4.8")
ar.eval_results["Over"]["Precision calculation"]


Use this output to inspect how observed misstatements are converted into an upper misstatement bound. The audit interpretation is whether that bound remains below the performance materiality used in planning.
The upper bounds $M_U$ in Table 4.5 are calculated next.


In [ ]:
display(upper_bound(k=0, popn=13500000, n=145, alpha=0.05))
display(upper_bound(k=1, popn=13500000, n=145, alpha=0.05))
display(upper_bound(k=2, popn=13500000, n=145, alpha=0.05))
display(upper_bound(k=3, popn=13500000, n=145, alpha=0.05))


## Exercise 4.9. Cell evaluation


This exercise evaluates the same audit evidence with the cell evaluation method. The practical task is to compare how a different MUS evaluation method turns the observed errors into a precision calculation.
The results of the cell evaluation method, as presented in Table 4.8, can also be obtained by changing the evaluation method.


In [ ]:
ada_set_context("4.9")
display(pd.set_option("display.width", 70))
display(ar.evaluate(av=myResults["av"], evalMeth="cell"))
display(ar.eval_results["Over"]["Precision calculation"])


Compare this precision calculation with the Stringer output from Exercise 4.8. Note the large difference in achieved precision between the two evaluation methods: 354,168 for the Stringer bound versus 458,352 for the cell evaluation. The cell evaluation is more conservative here because at each stage the most conservative of the simple-spread and load-and-spread bounds is chosen. The important question is whether the evaluation method changes the upper bound enough to affect the supported conclusion.


## Exercise 4.10. PPS estimation


Finally, we present results from the `pps` evaluation method. For this purpose, we use audit values stored in the variable `av2`. The total error amount reflected in `av2` is 450,000.


In [ ]:
ada_set_context("4.10")
myResults["av2"] = accounts_receivable.set_index("invoice").loc[myResults["item"], "av2"].to_numpy()
display(ar.evaluate(av=myResults["av2"], evalMeth="pps"))
display(pd.Series({"Error estimate": float(np.ravel(ar.eval_results["Error estimate"])[0])}))


Read the error estimate as the point estimate of misstatement under this evaluation method. A two-sided prediction interval around the PPS estimate is calculated according to Equations 4.6 and 4.7.


In [ ]:
ada_set_context("4.10")
pps_estimate_value = float(np.ravel(ar.eval_results["pps estimate"])[0])
pps_lower_value = float(np.ravel(ar.eval_results["Lower bound"])[0])
pps_upper_value = float(np.ravel(ar.eval_results["Upper bound"])[0])
display(pd.Series({"pps estimate": pps_estimate_value, "Lower bound": pps_lower_value, "Upper bound": pps_upper_value}))


Compare the interval with the point estimate. The practical question is not only what the most likely amount is, but how much uncertainty remains around that estimate when forming a supported conclusion.
